Obtain ID's of records that have zero subjects.  
Then free to experiment with methods to enrich their metadata.

In [47]:
NO_RECORDS = 150

In [48]:
from datetime import datetime, timezone
from json import JSONDecodeError
from logging.config import dictConfig
from typing import Any, Optional
import psycopg
from psycopg import errors as psycopg_errors
from psycopg.rows import dict_row
from config.logging_config import LOGGING_CONFIG
from config.postgres_config import PostgresConfig
from tasks import transform_batch, add_file_metadata
import os
from fastapi import FastAPI, Query, HTTPException
import logging
from pydantic import BaseModel, Field
from utils.queue_utils import HarvestEventQueue, detect_identifier_type

from lxml import etree as ET
# API calls
import requests


# added for these experiments
from utils.enrichment_methods import openaire_doi_get_info, _opeanaire_get_subjects

In [49]:
dictConfig(LOGGING_CONFIG)
logger = logging.getLogger(__name__)

In [50]:
user = os.getenv("POSTGRES_USER")
passwd = os.getenv("POSTGRES_PASSWORD")
host = os.getenv("POSTGRES_ADDRESS", "127.0.0.1")

connection_params = {
    "user" : user,
    "password" : passwd,
    "dbname" : "datasetdb",
    "port" : 5432,
    "host" : host
}

In [51]:
query_records = """
            -- Exploring subjects from record raw metadata

-- Every xpath defines a potential XML structure with corresponding subject tag placement.
-- If raw_metadata conforms to ANY of these xpaths, subjects will be concatenated <subject> tags. (removing array_to_string() will return arrays)
-- If raw_metadata doesn't conform to any path (and probably has no subject tags), subjects will be ''


WITH query AS (
    SELECT
        records.*,
        array_to_string(
                xpath('oai:record/oai:metadata/datacite:resource/datacite:subjects/datacite:subject/text()', raw_metadata,
                    '{{datacite, http://datacite.org/schema/kernel-4},
                      {oai, http://www.openarchives.org/OAI/2.0/}}'
                )::text[] ||
				xpath('oai:record/oai:metadata/oaidc:oai_datacite/oaidc:payload/datacite:resource/datacite:subjects/datacite:subject/text()', raw_metadata,
                    '{{datacite, http://datacite.org/schema/kernel-4},
                      {oai, http://www.openarchives.org/OAI/2.0/},
                      {oaidc, http://schema.datacite.org/oai/oai-1.1/}}'
                )::text[] ||
				xpath('oai:record/oai:metadata/oai:resource/datacite:subjects/datacite:subject/text()', raw_metadata,
                    '{{datacite, http://datacite.org/schema/kernel-4},
                      {oai, http://www.openarchives.org/OAI/2.0/}}'
                )::text[] ||
				xpath('oai:record/oai:metadata/oad:oai_datacite/oad:payload/datacite:resource/datacite:subjects/datacite:subject/text()', raw_metadata,
				-- note the 3 in datacite
                    '{{datacite, http://datacite.org/schema/kernel-3},
					  {oai, http://www.openarchives.org/OAI/2.0/},
     				  {oad, http://schema.datacite.org/oai/oai-1.0/}}'
                )::text[] ||
				-- same path as previous but uses kernel-4 namespace
				xpath('oai:record/oai:metadata/oad:oai_datacite/oad:payload/datacite:resource/datacite:subjects/datacite:subject/text()', raw_metadata,
                    '{{datacite, http://datacite.org/schema/kernel-4},
					  {oai, http://www.openarchives.org/OAI/2.0/},
     				  {oad, http://schema.datacite.org/oai/oai-1.0/}}'
                )::text[]
				, ', '
        ) AS subjects
    FROM public.records
)
SELECT 
    id,
    title,
    raw_metadata,
    subjects
FROM query
WHERE query.subjects = ''
LIMIT %s
;
"""

In [52]:
try:
    with psycopg.connect(**connection_params, row_factory=dict_row) as conn:
        cur = conn.cursor()
        
        cur.execute(query_records, (NO_RECORDS,))
        
        results = cur.fetchall()
        print(results)

        record_ids = [str(row["id"]) for row in results]
        record_titles = [str(row["title"]) for row in  results]
        record_raw_metadata = [str(row["raw_metadata"]) for row in  results]


except Exception as e:
    logger.exception("Failed to fetch harvest runs")
    raise HTTPException(status_code=500, detail=str(e))

[{'id': '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/1476195659', 'title': 'iNEXT Discovery-Instruct ESRF-ERIC; Structural biology for translational research and discovery.', 'raw_metadata': '<record xmlns="http://www.openarchives.org/OAI/2.0/" xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance">\n  <header>\n    <identifier>oai:icatplus.esrf.fr:inv/1476195659</identifier>\n    <datestamp>2024-01-28T16:00:07.086Z</datestamp>\n  </header>\n  <metadata>\n    <oai_datacite xmlns="http://schema.datacite.org/oai/oai-1.1/" xsi:schemaLocation="http://schema.datacite.org/oai/oai-1.1/ http://schema.datacite.org/oai/oai-1.1/oai.xsd">\n      <schemaVersion>4.2</schemaVersion>\n      <datacentreSymbol>EI</datacentreSymbol>\n      <payload>\n        <resource xmlns="http://schema.datacite.org/meta/kernel-4.2/" xsi:schemaLocation="http://schema.datacite.org/meta/kernel-4.2/ http://schema.datacite.org/meta/kernel-4.2/metadata.xsd"><identifier identifierType="DOI">10.15151/ESRF-ES-1476195659</identifi

In [53]:
record_titles

['iNEXT Discovery-Instruct ESRF-ERIC; Structural biology for translational research and discovery.',
 'Bragg coherent modulation imaging with large sample-to-modulator distance',
 'Structural investigation of membrane and water-soluble proteins',
 'Structural Biology of Macromolecules and Macromolecular complexes from India',
 'Investigating the kinetics of dye-induced micellization of the cationic surfactant DTAB below its cmc with time-resolved SAXS',
 'Merging molten salt experiments with high pressure methods for synthesis of novel nanomaterials',
 'Investigating Fe-O-H reactions under variable oxidation conditions',
 'Structural Biology of Macromolecules and Macromolecular complexes from India',
 'In-situ observation of stress assisted grain boundary oxidation in Ni718',
 'In operando Structural Phase mapping of photosensitive neuristive devices under light excitation',
 'Structural Biology BAG - Portugal',
 'Breakdown of Stokes-Einstein: probing size-dependent viscosities in liqu

In [54]:
record_raw_metadata[10]

'<record xmlns="http://www.openarchives.org/OAI/2.0/" xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance">\n  <header>\n    <identifier>oai:icatplus.esrf.fr:inv/2233771726</identifier>\n    <datestamp>2025-11-06T00:00:13.427Z</datestamp>\n  </header>\n  <metadata>\n    <oai_datacite xmlns="http://schema.datacite.org/oai/oai-1.1/" xsi:schemaLocation="http://schema.datacite.org/oai/oai-1.1/ http://schema.datacite.org/oai/oai-1.1/oai.xsd">\n      <schemaVersion>4.2</schemaVersion>\n      <datacentreSymbol>EI</datacentreSymbol>\n      <payload>\n        <resource xmlns="http://schema.datacite.org/meta/kernel-4.2/" xsi:schemaLocation="http://schema.datacite.org/meta/kernel-4.2/ http://schema.datacite.org/meta/kernel-4.2/metadata.xsd"><identifier identifierType="DOI">10.15151/ESRF-ES-2233771726</identifier><creators><creator><creatorName nameType="Personal">Ana Luisa CARVALHO</creatorName><givenName>Ana Luisa</givenName><familyName>Carvalho</familyName><nameIdentifier nameIdentifierScheme=

In [55]:
def ET_from_string(string):
    return ET.ElementTree(ET.fromstring(string))

In [56]:
raw_xml = ET_from_string(record_raw_metadata[13])

In [57]:
raw_xml.getroot()

<Element {http://www.openarchives.org/OAI/2.0/}record at 0x1fff0772100>

In [58]:
#print(ET.tostring(raw_xml, pretty_print=True).decode())

In [59]:
# Finding a desired tag anywhere in tree, e.g. title
raw_xml.xpath("//*[local-name()='title']/text()")

['In situ studies of dendritic growth in Ga-In alloys']

In [60]:
element  = raw_xml.findall(".//{*}title")
print(element)



print(element[0].text)

[<Element {http://schema.datacite.org/meta/kernel-4.2/}title at 0x1fff07b5ac0>]
In situ studies of dendritic growth in Ga-In alloys


In [61]:
element  = raw_xml.findall(".//{*}identifier[@identifierType='DOI']")
element
#element[0].text

[<Element {http://schema.datacite.org/meta/kernel-4.2/}identifier at 0x1fff07b7ac0>]

In [62]:
raw_xml.xpath("//*[local-name()='date']/text()")

['2024-04-18T06:00:00Z/2024-04-22T06:00:00Z', '2024-04-22T06:00:00Z']

In [63]:
# getting the DOI of the dataset
# Later, find related DOIs, articles...

# xpath returns a list of strings if /text() is included

raw_xml.xpath("//*[local-name()='identifier'][@identifierType='DOI']/text()")

['10.15151/ESRF-ES-1553175561']

In [64]:
record_number = 6

print(ET.tostring(ET_from_string(record_raw_metadata[record_number - 1]), pretty_print=True).decode())

<record xmlns="http://www.openarchives.org/OAI/2.0/" xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance">
  <header>
    <identifier>oai:icatplus.esrf.fr:inv/1226443260</identifier>
    <datestamp>2023-07-22T06:00:21.083Z</datestamp>
  </header>
  <metadata>
    <oai_datacite xmlns="http://schema.datacite.org/oai/oai-1.1/" xsi:schemaLocation="http://schema.datacite.org/oai/oai-1.1/ http://schema.datacite.org/oai/oai-1.1/oai.xsd">
      <schemaVersion>4.2</schemaVersion>
      <datacentreSymbol>EI</datacentreSymbol>
      <payload>
        <resource xmlns="http://schema.datacite.org/meta/kernel-4.2/" xsi:schemaLocation="http://schema.datacite.org/meta/kernel-4.2/ http://schema.datacite.org/meta/kernel-4.2/metadata.xsd"><identifier identifierType="DOI">10.15151/ESRF-ES-1226443260</identifier><creators><creator><creatorName nameType="Personal">Leon ANDRIAMBARIARIJAONA</creatorName><givenName>Leon</givenName><familyName>Andriambariarijaona</familyName></creator><creator><creatorName name

In [65]:
rec = ET_from_string(record_raw_metadata[5])

In [66]:
rec.xpath("//*[local-name()='identifier'][@identifierType='DOI']/text()")[0]

'10.15151/ESRF-ES-1226443260'

# DOI check through bigger repositories  (#1)
Take an XML ETree.
Searches for identifier with identifierType DOI.
    - what if there are multiple, and one is under "related works"?
Seaches thru OpenAlex or Openaire - apis?
Data available? N -> end
Y ->
    search for fields: subject, topic, field, discipline, keyword

OpenAIRE API
- they're officialy moving to a Graph API in May 2026
https://api.openaire.eu/graph/swagger-ui/index.html

In [67]:
rec = ET_from_string(record_raw_metadata[9])
doi = rec.xpath("//*[local-name()='identifier'][@identifierType='DOI']/text()")

if doi == []:
    print("NO DOI")
else:
    print(openaire_doi_get_info(doi[0]))

DEBUG:    2026-05-25 14:02:42 urllib3.connectionpool@1049 Starting new HTTPS connection (1): api.openaire.eu:443
DEBUG:    2026-05-25 14:02:42 urllib3.connectionpool@544 https://api.openaire.eu:443 "GET /graph/v2/researchProducts?pid=10.15151%2FESRF-ES-1851552819 HTTP/1.1" 200 None
{'authors': [{'fullName': 'Miguel Angel GARCIA-TUNON', 'name': 'Miguel Angel', 'surname': 'Garcia-Tunon', 'rank': 1, 'pid': None}, {'fullName': 'Oscar RODRIGUEZ DE LA FUENTE', 'name': 'Oscar', 'surname': 'Rodriguez La Fuente', 'rank': 2, 'pid': None}, {'fullName': 'Alvaro MUNOZ NOVAL', 'name': 'Alvaro', 'surname': 'Munoz Noval', 'rank': 3, 'pid': None}, {'fullName': 'Manuel ABUIN', 'name': 'Manuel', 'surname': 'Abuin', 'rank': 4, 'pid': {'id': {'scheme': 'orcid', 'value': '0000-0002-5058-3262'}, 'provenance': None}}, {'fullName': 'Ugwumsinachi OJI', 'name': 'Ugwumsinachi', 'surname': 'Oji', 'rank': 5, 'pid': None}, {'fullName': 'Jaime José DOLADO FERNANDEZ', 'name': 'Jaime José', 'surname': 'Dolado Fernande

In [68]:
# get only records that have a DOI
ix_has_doi = []

for i in range(0, len(record_raw_metadata)):
    rec = ET_from_string(record_raw_metadata[i])
    doi = rec.xpath("//*[local-name()='identifier'][@identifierType='DOI']/text()")
    if doi != []:
        ix_has_doi.append(i)

print(ix_has_doi)

[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99, 100, 101, 102, 103, 104, 105, 106, 107, 108, 109, 110, 111, 112, 113, 114, 115, 116, 117, 118, 119, 120, 121, 122, 123, 124, 125, 126, 127, 128, 129, 130, 131, 132, 133, 134, 135, 136, 137, 138, 139, 140, 141, 142, 143, 144, 145, 146, 147, 148, 149]


In [69]:
import random
import time

results = []
for i in ix_has_doi:
    rec = ET_from_string(record_raw_metadata[i])
    doi = rec.xpath("//*[local-name()='identifier'][@identifierType='DOI']/text()")

    if doi == []:
        print(f"{i} : NO DOI")
    else:
        doi = doi[0]

        # small pause to not overwhelm API
        time.sleep(random.uniform(0.01, 0.5))

        print(f"DOI for record {i}")
        try:
            res = openaire_doi_get_info(doi)
        except Exception as e:
            print(f"ERROR: Record {i} met with error: {str(e)}")
            res = {}

        print(res)
        results.append(res)

DOI for record 0
DEBUG:    2026-05-25 14:02:43 urllib3.connectionpool@1049 Starting new HTTPS connection (1): api.openaire.eu:443
DEBUG:    2026-05-25 14:02:43 urllib3.connectionpool@544 https://api.openaire.eu:443 "GET /graph/v2/researchProducts?pid=10.15151%2FESRF-ES-1476195659 HTTP/1.1" 200 None
{'authors': [{'fullName': 'Jose Antonio MARQUEZ', 'name': 'Jose Antonio', 'surname': 'Marquez', 'rank': 1, 'pid': None}, {'fullName': 'David FLOT', 'name': 'David', 'surname': 'Flot', 'rank': 2, 'pid': {'id': {'scheme': 'orcid_pending', 'value': '0000-0001-6639-7205'}, 'provenance': None}}], 'openAccessColor': None, 'publiclyFunded': None, 'type': 'dataset', 'language': {'code': 'eng', 'label': 'English'}, 'countries': None, 'subjects': [{'subject': {'scheme': 'keyword', 'value': 'PaN'}, 'provenance': None}, {'subject': {'scheme': 'keyword', 'value': 'MX-2524'}, 'provenance': None}, {'subject': {'scheme': 'keyword', 'value': 'ID30B'}, 'provenance': None}, {'subject': {'scheme': 'keyword', 'v

In [70]:
results[6]

{'authors': [{'fullName': 'Ninel SHARAPOVA',
   'name': 'Ninel',
   'surname': 'Sharapova',
   'rank': 1,
   'pid': None},
  {'fullName': 'Anna PAKHOMOVA',
   'name': 'Anna',
   'surname': 'Pakhomova',
   'rank': 2,
   'pid': None},
  {'fullName': 'Valentin KOVALEV',
   'name': 'Valentin',
   'surname': 'Kovalev',
   'rank': 3,
   'pid': {'id': {'scheme': 'orcid_pending', 'value': '0009-0000-0802-4179'},
    'provenance': None}},
  {'fullName': 'Maxim BYKOV',
   'name': 'Maxim',
   'surname': 'Bykov',
   'rank': 4,
   'pid': {'id': {'scheme': 'orcid_pending', 'value': '0000-0003-0248-1728'},
    'provenance': None}},
  {'fullName': 'Pascal Lennert JURZICK',
   'name': 'Pascal Lennert',
   'surname': 'Jurzick',
   'rank': 5,
   'pid': {'id': {'scheme': 'orcid_pending', 'value': '0009-0006-3044-3955'},
    'provenance': None}},
  {'fullName': 'Fariia Iasmin AKBAR',
   'name': 'Fariia Iasmin',
   'surname': 'Akbar',
   'rank': 6,
   'pid': {'id': {'scheme': 'orcid_pending', 'value': '0000

In [71]:
result_subjects = [_opeanaire_get_subjects(rez) for rez in results]


In [72]:
result_subjects[20]

[{'subject': {'scheme': 'keyword', 'value': 'PaN'}, 'provenance': None},
 {'subject': {'scheme': 'keyword', 'value': 'ID13'}, 'provenance': None},
 {'subject': {'scheme': 'keyword', 'value': 'Particles, Nuclei and Fields'},
  'provenance': None},
 {'subject': {'scheme': 'keyword', 'value': 'Life Sciences'},
  'provenance': None},
 {'subject': {'scheme': 'keyword', 'value': 'LS-3299'}, 'provenance': None}]

In [73]:
ix = ix_has_doi[20]
record_raw_metadata[ix]

'<record xmlns="http://www.openarchives.org/OAI/2.0/" xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance">\n  <header>\n    <identifier>oai:icatplus.esrf.fr:inv/1438786078</identifier>\n    <datestamp>2024-02-19T07:00:23.982Z</datestamp>\n  </header>\n  <metadata>\n    <oai_datacite xmlns="http://schema.datacite.org/oai/oai-1.1/" xsi:schemaLocation="http://schema.datacite.org/oai/oai-1.1/ http://schema.datacite.org/oai/oai-1.1/oai.xsd">\n      <schemaVersion>4.2</schemaVersion>\n      <datacentreSymbol>EI</datacentreSymbol>\n      <payload>\n        <resource xmlns="http://schema.datacite.org/meta/kernel-4.2/" xsi:schemaLocation="http://schema.datacite.org/meta/kernel-4.2/ http://schema.datacite.org/meta/kernel-4.2/metadata.xsd"><identifier identifierType="DOI">10.15151/ESRF-ES-1438786078</identifier><creators><creator><creatorName nameType="Personal">Tilman GRUNEWALD</creatorName><givenName>Tilman</givenName><familyName>Grunewald</familyName><nameIdentifier nameIdentifierScheme="ORC

In [74]:
openaire_doi_get_info("10.15151/ESRF-ES-1101291159")

DEBUG:    2026-05-25 14:05:02 urllib3.connectionpool@1049 Starting new HTTPS connection (1): api.openaire.eu:443
DEBUG:    2026-05-25 14:05:03 urllib3.connectionpool@544 https://api.openaire.eu:443 "GET /graph/v2/researchProducts?pid=10.15151%2FESRF-ES-1101291159 HTTP/1.1" 200 None


{'authors': [{'fullName': 'Wanner, Adrian',
   'name': 'Adrian',
   'surname': 'Wanner',
   'rank': 1,
   'pid': None}],
 'openAccessColor': None,
 'publiclyFunded': None,
 'type': 'dataset',
 'language': {'code': 'eng', 'label': 'English'},
 'countries': None,
 'subjects': [{'subject': {'scheme': 'keyword', 'value': 'Life Sciences'},
   'provenance': None},
  {'subject': {'scheme': 'keyword', 'value': 'LS-3193'}, 'provenance': None},
  {'subject': {'scheme': 'keyword', 'value': 'ID16A'}, 'provenance': None}],
 'mainTitle': 'Structure and function of neuronal circuits underlying working memory-related sequential activity',
 'subTitle': None,
 'descriptions': ['Working memory is thought to be mediated by stereotypic neuronal activity patterns in the brain such as neural sequences, in which subsets of neurons are sequentially active in a stereotypical temporal order. Neuronal interactions are fundamentally constrained by the synaptic wiring diagram, but the connectivity between neurons t

In [75]:
result_subjects

[[{'subject': {'scheme': 'keyword', 'value': 'PaN'}, 'provenance': None},
  {'subject': {'scheme': 'keyword', 'value': 'MX-2524'}, 'provenance': None},
  {'subject': {'scheme': 'keyword', 'value': 'ID30B'}, 'provenance': None},
  {'subject': {'scheme': 'keyword', 'value': 'Particles, Nuclei and Fields'},
   'provenance': None},
  {'subject': {'scheme': 'keyword', 'value': 'ID30A-1'}, 'provenance': None},
  {'subject': {'scheme': 'keyword', 'value': 'Macromolecular Crystallography'},
   'provenance': None}],
 [{'subject': {'scheme': 'keyword',
    'value': 'In-house Method and Instrumentation'},
   'provenance': None},
  {'subject': {'scheme': 'keyword', 'value': 'ID01'}, 'provenance': None},
  {'subject': {'scheme': 'keyword', 'value': 'IH-MI-1606'},
   'provenance': None}],
 [{'subject': {'scheme': 'keyword', 'value': 'BM07'}, 'provenance': None},
  {'subject': {'scheme': 'keyword', 'value': 'ID30A-1'}, 'provenance': None},
  {'subject': {'scheme': 'keyword', 'value': 'ID30A-3'}, 'pro

In [76]:
len(result_subjects)

150

In [77]:
len([result for result in result_subjects if result != []])

142

In [78]:
from utils.enrichment_methods import openaire_doi_get_info, _opeanaire_get_subjects

res = openaire_doi_get_info("10.15151/ESRF-ES-632216926")
print(res)
res_subjects = _opeanaire_get_subjects(res)
print(res_subjects)

DEBUG:    2026-05-25 14:05:03 urllib3.connectionpool@1049 Starting new HTTPS connection (1): api.openaire.eu:443
DEBUG:    2026-05-25 14:05:04 urllib3.connectionpool@544 https://api.openaire.eu:443 "GET /graph/v2/researchProducts?pid=10.15151%2FESRF-ES-632216926 HTTP/1.1" 200 None
{'authors': [{'fullName': 'Amrita PATEL', 'name': 'Amrita', 'surname': 'Patel', 'rank': 1, 'pid': None}, {'fullName': 'Klaus FUTTERER', 'name': 'Klaus', 'surname': 'Futterer', 'rank': 2, 'pid': None}, {'fullName': 'Didier NURIZZO', 'name': 'Didier', 'surname': 'Nurizzo', 'rank': 3, 'pid': {'id': {'scheme': 'orcid_pending', 'value': '0000-0002-7367-5098'}, 'provenance': None}}, {'fullName': 'Daniel PANNE', 'name': 'Daniel', 'surname': 'Panne', 'rank': 4, 'pid': {'id': {'scheme': 'orcid_pending', 'value': '0000-0001-9158-5507'}, 'provenance': None}}], 'openAccessColor': None, 'publiclyFunded': None, 'type': 'dataset', 'language': {'code': 'eng', 'label': 'English'}, 'countries': None, 'subjects': [{'subject': {

In [79]:
# Mangled, fake DOI sure to get no results:

res = openaire_doi_get_info("123456FALSE")
print(res)
res_subjects = _opeanaire_get_subjects(res)
print(res_subjects)

DEBUG:    2026-05-25 14:05:04 urllib3.connectionpool@1049 Starting new HTTPS connection (1): api.openaire.eu:443
DEBUG:    2026-05-25 14:05:04 urllib3.connectionpool@544 https://api.openaire.eu:443 "GET /graph/v2/researchProducts?pid=123456FALSE HTTP/1.1" 200 None
{}
[]


# DOI resolver method  (#2)
Take an XML ETree.
Searches for identifier with identifierType DOI.
    - what if there are multiple, and one is under "related works"?
Talks to an API such as DataCite or SignPosting.
Data available? N -> end
Y ->
    search for fields: subject, topic, field, discipline, keyword